In [5]:
from imagematerials.rest_of.biomass import *

from imagematerials.rest_of.const import path_scenario_data, SCENARIO

In [6]:
# Crop consumption in Gg dm/yr, per type of use and crop type including other crops. Dimensions:  [5,17,27] (t) , [NUFPT, NFCT, NRT](time)
crops_cons = read_mym_df(path_scenario_data + f'AGRCONSCTF_DM.OUT').set_index(["time", "DIM_1", "DIM_2"])  

# Wood demand per woodtype in 1000m3/yr. Dimensions: [4,27](t), [NWCT,NRT] (time)
wood_demand = read_mym_df(path_scenario_data + 'WDEMAND.OUT').set_index(["time", "DIM_1"])

# Feed consumption per grazing system type, feed product type and animal type in Gg dm/yr Dimensions:  [3,6,6,27] (t), [NGST,NFPT,NAT,NRT] (t)
feed_cons = read_mym_df(path_scenario_data + 'TFEED.OUT').set_index(["time", "DIM_1", "DIM_2", "DIM_3"])

# Animal products  Unit=Gg dm/yr; Label=Consumption of animal products in dry matter, per type of use and animal type [5,6,27] (t) [NUFPT,NAPT,NRT] 
animal_products_cons = read_mym_df(path_scenario_data + 'AGRCONSA_DM.OUT').set_index(["time", "DIM_1", "DIM_2"])

# Biofuel crops production (same as consumption) difference is only made in energy trade, not for actual crops calculation
# Unit= Gg dm/yr; Label= Production of biofuels (dry matter) [5,27] [NBCT,NRT] (t)
biofuel_crops = read_mym_df(path_scenario_data + 'AGRPRODBF_dm.OUT').set_index(["time", "DIM_1"])

# Materials buildings (BUMA output) in kt
buildings_mat = pd.read_csv(path_input_data + f'/IMAGE_MAT_out/{SCENARIO}/material_output_buma_RASMI.csv', header = 0).set_index(['Unnamed: 0', 'flow', 'type', 'area', 'material'])

# Split up different biomass types 
# Crops: Split up crops

splitted_up_crops_total = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                            selected_DIM1= 'total', biomass_type = 'crops')
splitted_up_crops_food = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'food', biomass_type = 'crops')
splitted_up_crops_feed = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'feed', biomass_type = 'crops')
splitted_up_crops_other_use = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'other use', biomass_type = 'crops')
# Feed
splitted_up_feed = split_up_feed(feed_cons)

# Wood & wood from buildings
splitted_up_wood = split_up_wood(wood_demand, buildings_mat)

# Animal Products
splitted_up_animal_products = split_up_animal_products(animal_products_cons)

# Biofuel crops 
splitted_up_biofuel_crops = split_up_biofuelcrops(biofuel_crops)

#%% get global values

crops_total_consumption_global = sum_by_region(splitted_up_crops_total)
feed_total_consumption_global = sum_by_region(splitted_up_feed)


In [7]:
crops_total_consumption_global

,sugar crops,cereals,vegetables and fruits,oil bearing crops,roots and tubers,rice,wheat,pulses,spices etc.,fibres,total
time,,,,,,,,,,,
1970,0.218248,0.261524,0.301174,0.095924,0.154732,0.269516,0.288431,0.092142,0.025400,0.044291,1.751381
1971,0.224385,0.265802,0.312883,0.098183,0.158152,0.279047,0.296445,0.096236,0.025931,0.045280,1.802344
1972,0.229397,0.274058,0.323185,0.099809,0.158138,0.282235,0.308201,0.097780,0.026040,0.046509,1.845353
1973,0.233609,0.271197,0.336847,0.102223,0.158106,0.289146,0.319180,0.102816,0.026925,0.045976,1.886025
1974,0.244000,0.273501,0.342158,0.103070,0.159501,0.294102,0.336581,0.104762,0.027149,0.045344,1.930168
...,...,...,...,...,...,...,...,...,...,...,...
2096,0.852563,0.515573,1.901926,0.795619,0.492589,0.904052,1.103296,0.617845,0.251846,0.243429,7.678737
2097,0.853586,0.517345,1.904181,0.796385,0.493607,0.903580,1.104919,0.617796,0.253205,0.244386,7.688991
2098,0.854603,0.519117,1.906422,0.797146,0.494625,0.903101,1.106538,0.617786,0.254564,0.245341,7.699243


In [ ]:
# Create Sankey diagram for total biomass consumption
# this does not run! 

# sankey_total, link_source_bio, link_target_bio, link_value_bio  = sankey_total_biomass(splitted_up_crops_food, 
#                         splitted_up_crops_feed, 
#                         splitted_up_biofuel_crops, 
#                         splitted_up_animal_products, 
#                         splitted_up_wood, 
#                         splitted_up_feed, 
#                         'figures_test',  # path to save the figures
#                         2060, 27)